# Day 03 프로젝트 실습
## 건강한하루 RAG 시스템 만들기

> 이 노트북은 아래 세 자료를 하나로 잇는 실습입니다.
>
> - **[Day04(Week02) SQLite로 RDB 구축하고 데이터 저장하기](../../../week02_데이터전처리_RDB설계/day04_4일차/프로젝트/SQLite로%20RDB%20구축하고%20데이터%20저장하기.ipynb)** 에서 만든 `건강한하루.db`를 그대로 사용합니다.
> - **[RAG 데이터 설계](../RAG%20데이터%20설계.md)** 에서 배운 Chunk·메타데이터 설계 기준을 실제 DB 데이터에 적용합니다.
> - **[RAG 예제](../RAG%20예제/)** 의 Chroma + LangChain LCEL 코드 패턴을 그대로 재사용합니다.

### 오늘 만드는 것
1. `data/건강한하루.db`의 3개 테이블(상품, 영양소_기능성, 영양소_섭취기준)을 불러온다.
2. 각 행을 "하나의 질문에 답할 수 있는" 단위의 문서(Chunk)로 만든다.
3. 로컬 Ollama 임베딩으로 벡터화하고 Chroma 벡터저장소에 저장한다.
4. Groq API의 LLM과 LangChain LCEL로 RAG 체인을 만든다.
5. RF-3(AI 상담/RAG FAQ) 요구사항대로 **근거 문서를 인용하고, 근거가 없으면 모른다고 답하는** RAG 시스템을 테스트한다.


---
## 0. 준비: 라이브러리와 환경변수

`pyproject.toml`에 정의된 라이브러리(LangChain, Chroma, python-dotenv 등)를 사용합니다.

**LLM(답변 생성)에는 Groq API를 사용합니다.** 실행 전에 아래 과정을 먼저 진행하세요.

1. 이 폴더의 `.env.example`을 복사해 `.env` 파일을 만든다.
2. [Groq API Key 발급](https://console.groq.com/keys) 후 `.env`의 `GROQ_API_KEY`에 붙여넣는다.

**임베딩(문서 벡터화)에는 로컬 Ollama 모델을 사용하므로 별도의 API 키가 필요하지 않습니다.**
대신 아래 명령으로 미리 모델을 받아두고, `ollama serve`로 서버를 띄워두어야 합니다.

```
ollama pull qwen3-embedding:0.6b
```


In [1]:
import sqlite3

import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document

# .env 파일의 GROQ_API_KEY를 환경변수로 불러옵니다.
load_dotenv()

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


---
## 1. SQLite에서 데이터 불러오기

Week02 Day04에서 설계한 3개 테이블을 그대로 조회합니다. 빈 값(NaN)은 "정보 없음"으로 채워 문서 안에 어색한 값이 섞이지 않게 합니다.


In [2]:
conn = sqlite3.connect("data/건강한하루.db")

product = pd.read_sql_query("SELECT * FROM 상품", conn).fillna("정보 없음")
func_db = pd.read_sql_query("SELECT * FROM 영양소_기능성", conn).fillna("정보 없음")
rda_db = pd.read_sql_query("SELECT * FROM 영양소_섭취기준", conn).fillna("정보 없음")

conn.close()

print("상품:", product.shape)
print("영양소_기능성:", func_db.shape)
print("영양소_섭취기준:", rda_db.shape)


상품: (44, 15)
영양소_기능성: (54, 11)
영양소_섭취기준: (228, 14)


---
## 2. RAG 문서(Document) 설계

**[RAG 데이터 설계](../RAG%20데이터%20설계.md)** 의 두 가지 기준을 그대로 적용합니다.

- **Chunk 기준** : "하나의 Chunk는 하나의 질문에 답할 만큼 충분해야 한다" → 테이블의 한 행(row)을 하나의 문서로 만들되, 제목과 항목을 구분해 정리합니다. (조건 기준 청킹: 상품별, 영양소·연령대·성별 조건별로 이미 행이 나뉘어 있어 그대로 좋은 Chunk 단위가 됩니다.)
- **메타데이터** : 검색 필터링과 출처 표시에 쓸 수 있도록 `source`(데이터 종류), 식별 정보(상품명/영양소 등)를 함께 저장합니다.

3개 테이블 → 3종류의 문서를 만듭니다.


In [3]:
def build_product_documents(df: pd.DataFrame) -> list[Document]:
    """상품마스터 한 행 = 상품 하나에 대한 Chunk"""
    docs = []
    for _, row in df.iterrows():
        content = (
            f"[상품 정보] {row['상품명']} ({row['카테고리']})\n"
            f"- 업소명: {row['업소명']}\n"
            f"- 주요 원료: {row['검색키워드(주요원료)']}\n"
            f"- 관련 증상: {row['관련_증상']}\n"
            f"- 섭취량 및 섭취방법: {row['섭취량_섭취방법']}\n"
            f"- 섭취 시 주의사항: {row['섭취시주의사항']}\n"
            f"- 기능성 내용: {row['기능성_내용']}"
        )
        metadata = {
            "source": "상품마스터",
            "상품ID": row["상품ID"],
            "상품명": row["상품명"],
            "카테고리": row["카테고리"],
            "주요원료": row["검색키워드(주요원료)"],
            "데이터출처": row["데이터출처"],
        }
        docs.append(Document(page_content=content, metadata=metadata))
    return docs


In [4]:
def build_function_documents(df: pd.DataFrame) -> list[Document]:
    """영양소_기능성 한 행 = 영양소 하나의 기능성 근거에 대한 Chunk"""
    docs = []
    for _, row in df.iterrows():
        content = (
            f"[영양소 기능성 정보] {row['영양소']} ({row['분류']})\n"
            f"- 기능성: {row['기능성']}\n"
            f"- 관련 증상: {row['관련_증상']}\n"
            f"- 근거 수준: {row['근거_수준']} (신뢰도: {row['신뢰도']})\n"
            f"- 근거 설명: {row['근거_설명']}\n"
            f"- 출처: {row['출처']}"
        )
        metadata = {
            "source": "영양소_기능성_추천DB",
            "영양소": row["영양소"],
            "분류": row["분류"],
            "근거_수준": row["근거_수준"],
            "신뢰도": row["신뢰도"],
        }
        docs.append(Document(page_content=content, metadata=metadata))
    return docs


In [5]:
def build_intake_documents(df: pd.DataFrame) -> list[Document]:
    """영양소_섭취기준 한 행 = 영양소·연령대·성별 조건 하나에 대한 Chunk"""
    docs = []
    for _, row in df.iterrows():
        content = (
            f"[섭취기준 정보] {row['영양소']} - {row['연령대']} {row['성별']} "
            f"(임신여부: {row['임신여부']}, 수유여부: {row['수유여부']})\n"
            f"- 권장섭취량: {row['권장섭취량']} {row['단위']}\n"
            f"- 충분섭취량: {row['충분섭취량']} {row['단위']}\n"
            f"- 상한섭취량: {row['상한섭취량']} {row['단위']}\n"
            f"- 출처: {row['출처']} ({row['기관']})\n"
            f"- 비고: {row['비고']}"
        )
        metadata = {
            "source": "영양소별_권장섭취량",
            "영양소": row["영양소"],
            "연령대": row["연령대"],
            "성별": row["성별"],
            "임신여부": row["임신여부"],
            "수유여부": row["수유여부"],
        }
        docs.append(Document(page_content=content, metadata=metadata))
    return docs


In [6]:
product_docs = build_product_documents(product)
function_docs = build_function_documents(func_db)
intake_docs = build_intake_documents(rda_db)

all_docs = product_docs + function_docs + intake_docs

print(f"상품 문서: {len(product_docs)}개")
print(f"영양소_기능성 문서: {len(function_docs)}개")
print(f"영양소_섭취기준 문서: {len(intake_docs)}개")
print(f"전체 문서: {len(all_docs)}개")
print()
print("--- 문서 예시 ---")
print(all_docs[0].page_content)
print(all_docs[0].metadata)


상품 문서: 44개
영양소_기능성 문서: 54개
영양소_섭취기준 문서: 228개
전체 문서: 326개

--- 문서 예시 ---
[상품 정보] 뉴메릿 비타민C&D 메가 (비타민)
- 업소명: (주)푸드어셈블
- 주요 원료: 비타민 C
- 관련 증상: 피로
- 섭취량 및 섭취방법: 1일1회 , 1회 1포(2.1g)를 물과 함께 섭취하십시오
- 섭취 시 주의사항: 1. 고칼슘혈증이 있거나 의약품 복용 시 전문가와 상담할 것 / 2. 이상사례 발생 시 섭취를 중단하고 전문가와 상담할 것 / 3. 과량 섭취하지 않도록 주의할 것 / 4. 신장질환이 있는 경우 섭취 전 전문가와 상담할 것
- 기능성 내용: [비타민C] / (1) 결합조직 형성과 기능유지에 필요 / (2) 철의 흡수에 필요 / (3) 항산화 작용을 하여 유해산소로부터 세포를 보호하는데 필요 / [비타민D] / (1) 칼슘과 인이 흡수되고 이용되는데 필요 / (2) 뼈의 형성과 유지에 필요 / (3) 골다공증발생 위험 감소에 도움을 줌
{'source': '상품마스터', '상품ID': 'P001', '상품명': '뉴메릿 비타민C&D 메가', '카테고리': '비타민', '주요원료': '비타민 C', '데이터출처': '식품안전나라(foodsafetykorea.go.kr) 건강기능식품 검색'}


---
## 3. 임베딩 및 벡터저장소 구축 (Chroma)

**[RAG 예제 - 1. Create VectorDB](../RAG%20예제/1.%20Create%20VectorDB.ipynb)** 와 같은 방식으로, 로컬 Ollama 임베딩 모델(`qwen3-embedding:0.6b`)을 사용해 Chroma 벡터저장소를 만듭니다. 임베딩은 API 키가 필요 없으므로, 답변 생성(LLM)에만 Groq API를 쓰는 구조입니다.


In [7]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")


In [8]:
import shutil
from pathlib import Path

from langchain_chroma import Chroma

CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "healthy-daily-rag"

# 다시 실행해도 문서가 중복 저장되지 않도록, 기존에 만들어진 Chroma 디렉터리가 있으면 통째로 삭제하고 새로 만듭니다.
if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)

vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
)

print("벡터저장소 구축 완료!")
print(f"저장된 문서 수: {vectorstore._collection.count()}")

벡터저장소 구축 완료!
저장된 문서 수: 326


---
## 4. 벡터 검색 테스트

RAG 체인을 만들기 전에, 벡터저장소가 질문과 관련된 문서를 잘 찾아오는지 먼저 확인합니다.


In [9]:
test_queries = [
    "비타민C 관련 상품을 추천해줘",
    "비타민A 상한섭취량이 얼마인가요?",
    "임산부가 섭취해도 되는 영양소 기준이 있나요?",
]

for query in test_queries:
    print(f"\n질문: {query}")
    results = vectorstore.similarity_search(query, k=2)
    for r in results:
        print(f"  - [{r.metadata.get('source')}] {r.page_content[:60].replace(chr(10), ' ')}...")
    print("=" * 60)



질문: 비타민C 관련 상품을 추천해줘
  - [상품마스터] [상품 정보] 뉴메릿 비타민C&D 메가 (비타민) - 업소명: (주)푸드어셈블 - 주요 원료: 비타민 C -...
  - [영양소_기능성_추천DB] [영양소 기능성 정보] 비타민 C (비타민) - 기능성: 항산화 - 관련 증상: 피로 - 근거 수준: MFD...

질문: 비타민A 상한섭취량이 얼마인가요?
  - [영양소별_권장섭취량] [섭취기준 정보] 비타민 A - 50~64세 여성 (임신여부: 해당없음, 수유여부: 해당없음) - 권장섭취량...
  - [영양소별_권장섭취량] [섭취기준 정보] 비타민 A - 50~64세 남성 (임신여부: 해당없음, 수유여부: 해당없음) - 권장섭취량...

질문: 임산부가 섭취해도 되는 영양소 기준이 있나요?
  - [영양소별_권장섭취량] [섭취기준 정보] 철 - 임산부 여성 (임신여부: 예, 수유여부: 아니오) - 권장섭취량: 24.0 mg -...
  - [영양소별_권장섭취량] [섭취기준 정보] 비타민 D - 임산부 여성 (임신여부: 예, 수유여부: 아니오) - 권장섭취량: 정보 없음...


---
## 5. LLM 연결 및 RAG 체인 구성 (Groq)

여기서부터는 **`.env`에 실제 `GROQ_API_KEY`를 넣어야 실행됩니다.**

RF-3.0(AI 상담/RAG FAQ) 요구사항의 핵심 두 가지를 프롬프트에 그대로 반영합니다.

- **RF-3.3 근거 문서 인용** : 답변에 참고한 문서의 출처를 함께 보여준다.
- **[RAG 데이터 설계 - 정보가 없을 때의 처리](../RAG%20데이터%20설계.md)** : 관련 문서가 없으면 모르는 척 답하지 않고 "확인된 자료가 없다"고 안내한다. (Week04 ROI 검토의 "신뢰도 낮은 답변은 CS 이관" 원칙과 연결됩니다.)


In [10]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},  # 관련 문서 상위 3개를 참고
)

print("검색기 준비 완료!")


검색기 준비 완료!


In [11]:
from langchain_groq import ChatGroq

# GROQ_API_KEY를 사용하는 Groq 호스팅 모델로 답변을 생성합니다.
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=512,
)


#### LCEL을 위한 유틸리티 함수

검색된 문서를 프롬프트에 넣을 컨텍스트로 정리하고, 답변에 함께 보여줄 출처 목록을 만듭니다.


In [12]:
def format_docs_for_context(docs: list) -> str:
    """검색된 문서들을 프롬프트에 넣을 컨텍스트 문자열로 정리하는 함수"""
    formatted = []
    for i, doc in enumerate(docs, 1):
        formatted.append(f"[문서 {i}]\n{doc.page_content}")
    return "\n\n".join(formatted)


def extract_sources_from_docs(docs: list) -> list:
    """검색된 문서들에서 사용자에게 보여줄 출처 정보를 정리하는 함수 (중복 제거)"""
    labels = []
    for doc in docs:
        meta = doc.metadata
        if meta.get("상품명"):
            label = meta["상품명"]
        elif meta.get("영양소"):
            # 섭취기준 문서는 영양소만으로는 서로 구분되지 않으므로 연령대·성별도 함께 표기
            detail = " ".join(str(meta[k]) for k in ("연령대", "성별") if meta.get(k))
            label = f"{meta['영양소']} ({detail})" if detail else meta["영양소"]
        else:
            label = meta.get("source")
        labels.append(f"{meta.get('source')} - {label}")

    # 완전히 동일한 출처가 여러 번 검색된 경우 한 번만 표시
    unique_labels = list(dict.fromkeys(labels))
    return [f"[출처 {i}] {label}" for i, label in enumerate(unique_labels, 1)]

#### 프롬프트 템플릿

"컨텍스트에 없으면 모른다고 답한다"는 원칙을 프롬프트에 직접 명시합니다.


In [13]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
당신은 건강기능식품 쇼핑몰 '건강한하루'의 AI 상담원입니다.
아래 컨텍스트만 근거로 삼아 사용자 질문에 답변하세요.

지켜야 할 규칙:
1. 컨텍스트에 없는 내용은 답하지 말고, "확인된 자료가 없습니다. 정확한 안내가 필요하면 상담원 연결을 요청해주세요."라고 답하세요.
2. 특정 질환·복용약이 있는 경우 "전문가와 상담하라"는 주의사항이 컨텍스트에 있다면 반드시 함께 안내하세요.
3. 진단이나 처방처럼 단정적인 의학적 판단은 하지 마세요.
4. 반드시 한국어로, 간결하게 답변하세요.

컨텍스트:
{context}

질문:
{question}

답변:"""

prompt = PromptTemplate.from_template(template=prompt_template)


#### 최종 답변 포맷 함수

질문, 답변, 출처를 하나로 묶습니다. 다만 "확인된 자료가 없습니다"로 답한 경우에는 출처를 붙이지 않습니다.


In [14]:
def format_final_answer(inputs: dict) -> str:
    """질문, 답변, 출처를 정리해 최종 응답을 만드는 함수"""
    question = inputs["question"]
    answer = inputs["answer"]
    sources = inputs["sources"]

    result = f"질문: {question}\n답변: {answer}\n"

    if "확인된 자료가 없습니다" not in answer and sources:
        result += "출처: " + " / ".join(sources)

    return result


#### LCEL 체인 구성

**[RAG 예제 - 2. Create RAG](../RAG%20예제/2.%20Create%20RAG.ipynb)** 와 같은 LCEL 패턴을 그대로 사용합니다.


In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


def process_retrieved_docs(input_data: dict) -> dict:
    """검색된 문서를 처리하여 컨텍스트와 출처를 분리하는 함수"""
    question = input_data["question"]
    docs = input_data["context"]

    return {
        "question": question,
        "context": format_docs_for_context(docs),
        "sources": extract_sources_from_docs(docs),
    }


rag_chain = (
    {"question": RunnablePassthrough(), "context": retriever}
    | RunnableLambda(process_retrieved_docs)
    | {
        "answer": prompt | llm | StrOutputParser(),
        "question": lambda x: x["question"],
        "sources": lambda x: x["sources"],
    }
    | RunnableLambda(format_final_answer)
)

print("RAG 체인 구성 완료!")


RAG 체인 구성 완료!


---
## 6. RAG 시스템 테스트

RF-3에서 다뤄야 할 세 가지 유형의 질문을 모두 넣어봅니다.

- 상품 관련 질문 (상품마스터 문서로 답할 수 있어야 함)
- 섭취기준 관련 질문 (섭취기준 문서로 답할 수 있어야 함)
- 데이터에 없는 질문 (환불/배송처럼 준비되지 않은 주제 → "확인된 자료가 없습니다"로 답해야 함)


In [16]:
test_questions = [
    "비타민C가 들어간 상품을 추천해줘. 어떤 효과가 있어?",
    "임신 중에 철분은 하루에 얼마나 먹어야 해?",
    "이 쇼핑몰에서 주문한 상품 환불은 며칠 안에 가능해?",  # 컨텍스트에 없는 질문
]

for question in test_questions:
    try:
        answer = rag_chain.invoke(question)
        print(answer)
        print("-" * 80)
    except Exception as e:
        print(f"오류 발생: {e}")
        print("-" * 80)


질문: 비타민C가 들어간 상품을 추천해줘. 어떤 효과가 있어?
답변: 추천 상품  
1️⃣ **뉴메릿 비타민C&D 메가** – 1일 1포(2.1 g)  
2️⃣ **뉴메릿 비타민C&D 듀얼 메가** – 1일 1포(3.2 g)  

**효과**  
- 비타민 C: 항산화 작용으로 세포를 보호하고, 결합조직 형성·유지·철 흡수에 도움을 줍니다.  
- 비타민 D: 칼슘·인 흡수를 도와 뼈 형성과 유지, 골다공증 위험 감소에 기여합니다.  
- 두 제품 모두 “피로” 완화와 전반적인 활력 증진에 도움이 됩니다.  

**섭취 시 주의**  
고칼슘혈증, 약물 복용 중이거나 신장질환이 있는 경우 섭취 전 전문가와 상담하시기 바랍니다.  

필요하신 경우 상담원 연결을 요청해주세요.
출처: [출처 1] 상품마스터 - 뉴메릿 비타민C&D 메가 / [출처 2] 상품마스터 - 뉴메릿 비타민C&D 듀얼 메가 / [출처 3] 영양소_기능성_추천DB - 비타민 C
--------------------------------------------------------------------------------
질문: 임신 중에 철분은 하루에 얼마나 먹어야 해?
답변: 임신 중인 여성의 경우 하루 권장 철분 섭취량은 **24 mg**이며, 상한 섭취량은 **45 mg**입니다.
출처: [출처 1] 영양소별_권장섭취량 - 철 (임산부 여성) / [출처 2] 영양소별_권장섭취량 - 철 (수유부 여성) / [출처 3] 영양소별_권장섭취량 - 철 (30~49세 남성)
--------------------------------------------------------------------------------
질문: 이 쇼핑몰에서 주문한 상품 환불은 며칠 안에 가능해?
답변: 확인된 자료가 없습니다. 정확한 안내가 필요하면 상담원 연결을 요청해주세요.

------------------------------------------------------------------------------